In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

os.chdir("/content/drive/My Drive/베이지안자료분석PBL")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv("./data/pre_data/data.csv")

In [ ]:
from scipy.stats import shapiro, norm, probplot

shapiro_stat, shapiro_p = shapiro(data['close_pct'])

In [ ]:
shapiro_stat, shapiro_p

In [ ]:
print(f"Shapiro-Wilk Test: W-statistic = {shapiro_stat}, p-value = {shapiro_p}")
if shapiro_p > 0.05:
    print("Shapiro-Wilk Test 결과: 정규분포와 차이가 없습니다.")
else:
    print("Shapiro-Wilk Test 결과: 정규분포와 차이가 있습니다.")

In [ ]:
# 3. Q-Q Plot 시각화
plt.figure(figsize=(8, 6))
probplot(data['close_pct'], dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.grid(True)
plt.show()

In [ ]:
range_min = -10
range_max = 10
bins = 20
mu = 0
sigma = 1

bin_edges = np.linspace(range_min, range_max, bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2  # bin의 중앙값 계산
prior = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((bin_centers - mu) / sigma)**2)


# 결과 출력
print("Bin Centers:", bin_centers)
print("Prior Distribution:", prior_distribution)

# 히스토그램 시각화 (선택 사항)
plt.bar(bin_centers, prior_distribution, width=(range_max - range_min) / bins, alpha=0.7, edgecolor='black')
plt.title('Prior Distribution (Gaussian)')
plt.xlabel('Value')
plt.ylabel('Probability Density')
plt.grid()
plt.show()

In [ ]:
range_min = -10
range_max = 10
bins = 20
mu = 0.177
sigma = 3.75

bin_edges = np.linspace(range_min, range_max, bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2  # bin의 중앙값 계산
prior = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((bin_centers - mu) / sigma)**2)


# 결과 출력
print("Bin Centers:", bin_centers)
print("Prior Distribution:", prior)

# 히스토그램 시각화 (선택 사항)
plt.plot(bin_centers, prior, marker='o', linestyle='-', color='b')
plt.title('Prior Distribution (Gaussian)')
plt.xlabel('Value')
plt.ylabel('Probability Density')
plt.grid()
plt.show()

In [ ]:
data['close_pct'] *= 100

In [ ]:
prior

In [ ]:
data.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
data

In [ ]:
from sklearn.model_selection import train_test_split


def_data, update_data = train_test_split(data, test_size=0.5, random_state=42)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

# Assuming def_data['close_pct'] is a pandas Series
close_pct_mean = def_data['close_pct'].mean()
close_pct_std = def_data['close_pct'].std()

# Define the prior distribution
prior_distribution = norm(loc=close_pct_mean, scale=close_pct_std)

# Define the range and intervals from -10 to 10
num_bins = 20
close_pct_range = np.linspace(-10, 10, num_bins + 1)
theta_intervals = pd.IntervalIndex.from_breaks(close_pct_range, closed='right')

# Compute midpoints of intervals
theta_midpoints = np.array([interval.mid for interval in theta_intervals])

# Evaluate prior probabilities at midpoints and normalize
prior_probabilities = prior_distribution.pdf(theta_midpoints)
prior_probabilities /= prior_probabilities.sum()  # Normalize to ensure probabilities sum to 1

# Combine intervals and probabilities into a DataFrame
prior = pd.DataFrame({
    'theta_interval': [str(interval) for interval in theta_intervals],
    'prior_probability': prior_probabilities
})

# Plot the prior probabilities
prior['prior_probability'].plot(kind='bar', figsize=(10, 6), color='blue', alpha=0.7)
plt.xticks(ticks=np.arange(len(theta_intervals)), labels=prior['theta_interval'], rotation=90)
plt.xlabel('Theta Intervals')
plt.ylabel('Prior Probability')
plt.title('Prior Probability Distribution (-10 to 10)')
plt.show()


In [ ]:
def_data.shape

In [ ]:
import matplotlib.pyplot as plt

quantile_01 = def_data['close_pct'].quantile(0.0122)
quantile_99 = def_data['close_pct'].quantile(0.9832)

plt.figure(figsize=(10, 6))
def_data['close_pct'].hist(bins=100, alpha=0.7, color='blue')
plt.title('Histogram of close_pct')
plt.xlim(-15, 15)
plt.xlabel('close_pct')
plt.ylabel('Frequency')
plt.axvline(quantile_01, color='red', linestyle='dashed', linewidth=1, label='0.5% Quantile')
plt.axvline(quantile_99, color='green', linestyle='dashed', linewidth=1, label='99.5% Quantile')
plt.legend()
plt.show()


In [ ]:
from scipy.stats import percentileofscore

# 분위수 계산
percentile_10 = percentileofscore(def_data['close_pct'], 10.3) / 100

print(f"The value 10 corresponds to the {percentile_10:.4f} quantile.")

In [ ]:
def_data['close_pct'].describe()

In [ ]:
# 1. 구간 설정
bins = np.linspace(-3.5, 3.5, 8)  # -3.5 ~ 3.5 사이를 7개의 구간으로 나눔
labels = range(1, 8)  # 각 구간에 할당할 레이블

# 2. 각 컬럼에 대해 구간화 (Categorization)
def_data['PC1_bin'] = pd.cut(def_data['PC1'], bins=bins, labels=labels, include_lowest=True)
def_data['PC2_bin'] = pd.cut(def_data['PC2'], bins=bins, labels=labels, include_lowest=True)
def_data['PC3_bin'] = pd.cut(def_data['PC3'], bins=bins, labels=labels, include_lowest=True)

# 3. 결합확률 계산 (3D 히스토그램)
joint_prob, edges = np.histogramdd(
    def_data[['PC1_bin', 'PC2_bin', 'PC3_bin']].dropna().astype(int).values,
    bins=(7, 7, 7)
)

# 결합확률로 정규화
joint_prob /= joint_prob.sum()

# 4. 결합확률을 DataFrame으로 변환
# 모든 구간 조합의 인덱스 생성
pc1_indices, pc2_indices, pc3_indices = np.meshgrid(
    range(1, 8), range(1, 8), range(1, 8), indexing="ij"
)

# 다차원 배열을 1D로 펼쳐 DataFrame 생성
joint_prob_df = pd.DataFrame({
    'PC1_bin': pc1_indices.ravel(),
    'PC2_bin': pc2_indices.ravel(),
    'PC3_bin': pc3_indices.ravel(),
    'Probability': joint_prob.ravel()
})

# 5. 결과 확인
joint_prob_df

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 3D 시각화를 위한 데이터 준비
pc1 = joint_prob_df['PC1_bin']
pc2 = joint_prob_df['PC2_bin']
pc3 = joint_prob_df['PC3_bin']
probability = joint_prob_df['Probability']

# 3D 시각화
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 산점도 (확률 값을 색깔과 크기로 표현)
sc = ax.scatter(pc1, pc2, pc3, c=probability, cmap='viridis', s=probability * 1000, alpha=0.7)

# 컬러바 추가
cbar = plt.colorbar(sc, ax=ax, pad=0.1)
cbar.set_label('Probability Density')

# 축 레이블 설정
ax.set_xlabel('PC1_bin')
ax.set_ylabel('PC2_bin')
ax.set_zlabel('PC3_bin')
ax.set_title('3D Joint Probability Distribution')

plt.show()

In [ ]:
# 1. 구간 설정 (-10에서 10 사이를 20개의 구간으로 나눔)
bins = np.linspace(-10, 10, 21)  # 20개의 구간 경계
labels = range(1, 21)  # 각 구간에 할당할 레이블 (1부터 20까지)

# 2. 각 행을 구간에 따라 분류
def_data['bin'] = pd.cut(data['close_pct'], bins=bins, labels=labels, include_lowest=True)

# 3. 그룹화 및 요약
grouped = data.groupby('bin')['close_pct'].apply(list)

In [ ]:
def_data

In [ ]:
data.columns

In [ ]:
grouped_data = def_data.groupby('bin')[['PC1', 'PC2', 'PC3']]

In [ ]:
from scipy.stats import gaussian_kde


# Flatten PC1, PC2, PC3 data and calculate joint probability distribution
bins_pc = np.linspace(-3.5, 3.5, 8)  # -3.5 ~ 3.5를 7개의 구간으로 나눔
joint_probabilities = {}

for bin_label, group in grouped_data:
    # Compute 3D histogram for each bin group
    hist, edges = np.histogramdd(
        group[['PC1', 'PC2', 'PC3']].values,
        bins=(bins_pc, bins_pc, bins_pc)
    )
    # Normalize to create probability distribution
    hist_normalized = hist / hist.sum()
    joint_probabilities[bin_label] = hist_normalized

# 4. joint_probabilities 결과를 DataFrame으로 변환
# Convert the 3D histograms to a flat DataFrame for better analysis
final_df = []

for bin_label, prob_array in joint_probabilities.items():
    pc1_indices, pc2_indices, pc3_indices = np.meshgrid(
        range(1, 8), range(1, 8), range(1, 8), indexing="ij"
    )
    prob_flat = prob_array.ravel()
    df = pd.DataFrame({
        'bin': bin_label,
        'PC1_bin': pc1_indices.ravel(),
        'PC2_bin': pc2_indices.ravel(),
        'PC3_bin': pc3_indices.ravel(),
        'Probability': prob_flat
    })
    final_df.append(df)

result_df = pd.concat(final_df, ignore_index=True)

display(result_df)
result_df[result_df['Probability'] != 0]

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import numpy as np

# 3D 시각화
bins = result_df['bin'].unique()  # 고유한 bin 값 추출

for bin_label in bins:
    # 특정 bin 데이터 추출
    bin_data = result_df[result_df['bin'] == bin_label]

    # X, Y, Z 좌표 및 확률값
    pc1 = bin_data['PC1_bin'].astype(float)
    pc2 = bin_data['PC2_bin'].astype(float)
    pc3 = bin_data['PC3_bin'].astype(float)
    probabilities = bin_data['Probability']

    # 3D 좌표 데이터 준비
    positions = np.vstack([pc1, pc2, pc3])

    # 커널 밀도 추정 (weights로 PMF 반영)
    kde = gaussian_kde(positions, weights=probabilities)

    # 추정할 그리드 정의
    grid_pc1 = np.linspace(1, 7, 30)
    grid_pc2 = np.linspace(1, 7, 30)
    grid_pc3 = np.linspace(1, 7, 30)
    grid = np.array(np.meshgrid(grid_pc1, grid_pc2, grid_pc3)).reshape(3, -1)

    # PDF 추정
    pdf = kde(grid).reshape((30, 30, 30))

    # 3D 시각화
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # 3D 산점도 (밀도를 색으로 표현)
    sc = ax.scatter(grid_pc1, grid_pc2, grid_pc3, c=pdf.ravel(), cmap='viridis', alpha=0.6, s=5)

    # 컬러바 추가
    cbar = plt.colorbar(sc, ax=ax, pad=0.1)
    cbar.set_label('PDF Density')

    # 축 및 제목 설정
    ax.set_xlabel('PC1_bin')
    ax.set_ylabel('PC2_bin')
    ax.set_zlabel('PC3_bin')
    ax.set_title(f'3D PDF for Bin {bin_label}')

    plt.show()

In [ ]:
# 3D 시각화
for bin_label in bins:
    # 특정 bin 데이터 추출
    bin_data = result_df[result_df['bin'] == bin_label]

    # X, Y, Z 좌표 및 확률값
    pc1 = bin_data['PC1_bin'].astype(float)
    pc2 = bin_data['PC2_bin'].astype(float)
    pc3 = bin_data['PC3_bin'].astype(float)
    probabilities = bin_data['Probability']

    # 3D 좌표 데이터 준비
    positions = np.vstack([pc1, pc2, pc3])

    # 커널 밀도 추정 (weights로 PMF 반영)
    kde = gaussian_kde(positions, weights=probabilities)

    # 추정할 그리드 정의
    grid_pc1 = np.linspace(1, 7, 30)
    grid_pc2 = np.linspace(1, 7, 30)
    grid_pc3 = np.linspace(1, 7, 30)
    grid_pc1, grid_pc2, grid_pc3 = np.meshgrid(grid_pc1, grid_pc2, grid_pc3)
    grid = np.vstack([grid_pc1.ravel(), grid_pc2.ravel(), grid_pc3.ravel()])

    # PDF 추정
    pdf = kde(grid).reshape(grid_pc1.shape)

    # 3D 시각화
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # 3D 산점도
    sc = ax.scatter(
        grid_pc1.ravel(),
        grid_pc2.ravel(),
        grid_pc3.ravel(),
        c=pdf.ravel(),
        cmap='viridis',
        alpha=0.6,
        s=5
    )

    # 컬러바 추가
    cbar = plt.colorbar(sc, ax=ax, pad=0.1)
    cbar.set_label('PDF Density')

    # 축 및 제목 설정
    ax.set_xlabel('PC1_bin')
    ax.set_ylabel('PC2_bin')
    ax.set_zlabel('PC3_bin')
    ax.set_title(f'3D PDF for Bin {bin_label}')

    plt.show()
